# Predictor de Tendencias de Mercado con PythonTrabajo Final - Modulos y Paquetes para Machine Learning con PythonCaso practico: una empresa de analisis de datos financieros quiere predecir tendencias de mercado usando Machine Learning.

## 1. Cargar los datos

Importamos pandas para manejar la tabla de datos y numpy para operaciones numericas.

In [ ]:
import pandas as pdimport numpy as np

Leemos el csv con los datos de mercado que generamos con numpy (simulando precios de acciones).

In [ ]:
df = pd.read_csv('datos_mercado_financiero.csv', parse_dates=['fecha'])df.head()

Revisamos los tipos de dato y si hay valores nulos.

In [ ]:
df.info()

Estadisticas basicas de cada columna.

In [ ]:
df.describe()

Cuantos dias subio (1) y cuantos bajo (0) el precio.

In [ ]:
df['tendencia'].value_counts()

## 2. Visualizacion de los datos

matplotlib para graficos basicos, seaborn para graficos estadisticos.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as sns

Grafico de linea del precio de cierre a lo largo del tiempo.

In [ ]:
plt.figure(figsize=(10,4))plt.plot(df['fecha'], df['precio_cierre'])plt.xlabel('fecha')plt.ylabel('precio de cierre')plt.title('precio de cierre en el tiempo')plt.show()

Histograma para ver como se distribuyen los precios de cierre.

In [ ]:
plt.figure(figsize=(8,4))plt.hist(df['precio_cierre'], bins=30)plt.xlabel('precio de cierre')plt.ylabel('frecuencia')plt.title('histograma de precios de cierre')plt.show()

Grafico de dispersion, comparando volumen negociado contra precio de cierre.

In [ ]:
plt.figure(figsize=(8,4))plt.scatter(df['volumen'], df['precio_cierre'], c=df['tendencia'], cmap='coolwarm', alpha=0.6)plt.xlabel('volumen')plt.ylabel('precio de cierre')plt.title('volumen vs precio')plt.show()

Boxplot para comparar el volumen segun si el dia subio o bajo.

In [ ]:
plt.figure(figsize=(8,4))sns.boxplot(x='tendencia', y='volumen', data=df)plt.title('volumen segun tendencia')plt.show()

## 3. Modelo de clasificacion con Scikit-learn

DecisionTreeClassifier: el algoritmo de arbol de decision que vimos en clase con Iris y Wine.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

train_test_split: separa los datos en entrenamiento y prueba.

In [ ]:
from sklearn.model_selection import train_test_split

accuracy_score y classification_report: para medir que tan bien predice el modelo.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

Separamos las columnas de entrada (x) y la columna objetivo (y).

In [ ]:
x = df[['precio_apertura', 'precio_cierre', 'precio_max', 'precio_min', 'volumen']].valuesy = df['tendencia'].values

Dividimos en 80% entrenamiento y 20% prueba.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
print(x_train.shape)print(x_test.shape)

Creamos y entrenamos el modelo.

In [ ]:
model = DecisionTreeClassifier(random_state=42)model.fit(x_train, y_train)

Prediccion sobre los datos de prueba.

In [ ]:
predicciones = model.predict(x_test)

In [ ]:
print(predicciones)

In [ ]:
print(y_test)

Exactitud del modelo.

In [ ]:
accuracy_score(y_test, predicciones)

In [ ]:
print(classification_report(y_test, predicciones, target_names=['baja', 'sube']))

## 4. Red neuronal con PyTorch

torch y nn para construir la red. StandardScaler para normalizar los datos antes de meterlos a la red.

In [ ]:
import torchimport torch.nn as nnfrom sklearn.preprocessing import StandardScaler

Normalizamos los datos (igual que hicimos con Iris en clase).

In [ ]:
scaler = StandardScaler()x_scaled = scaler.fit_transform(x)

In [ ]:
x_train_t, x_test_t, y_train_t, y_test_t = train_test_split(x_scaled, y, test_size=0.2, random_state=42)

Pasamos los datos a tensores de torch.

In [ ]:
x_train_t = torch.tensor(x_train_t, dtype=torch.float32)x_test_t = torch.tensor(x_test_t, dtype=torch.float32)y_train_t = torch.tensor(y_train_t, dtype=torch.long)y_test_t = torch.tensor(y_test_t, dtype=torch.long)

Armamos la red: 5 entradas (las columnas), una capa oculta con ReLU, 2 salidas (sube o baja).

In [ ]:
modelo_nn = nn.Sequential(    nn.Linear(5, 16),    nn.ReLU(),    nn.Linear(16, 2))

Entropia cruzada porque es clasificacion, Adam como optimizador.

In [ ]:
loss_fn = nn.CrossEntropyLoss()optimizer = torch.optim.Adam(modelo_nn.parameters(), lr=0.01)

Entrenamos por 100 epocas.

In [ ]:
for epoch in range(100):    salida = modelo_nn(x_train_t)    loss = loss_fn(salida, y_train_t)    optimizer.zero_grad()    loss.backward()    optimizer.step()

In [ ]:
print(loss.item())

Probamos el modelo con los datos de prueba.

In [ ]:
with torch.no_grad():    predicciones_nn = modelo_nn(x_test_t).argmax(1)

In [ ]:
aciertos = (predicciones_nn == y_test_t).sum().item()total = len(y_test_t)aciertos / total

## 5. NLP - tokenizacion y TF-IDF

Reportes de ejemplo, como los que se analizarian de una empresa real.

In [ ]:
reportes = [    'las acciones suben tras buenos resultados',    'el mercado cae por incertidumbre economica',    'la empresa reporta ganancias record este trimestre']

Paso 1: tokenizar (separar cada frase en palabras).

In [ ]:
tokens = [texto.split() for texto in reportes]tokens

Paso 2: armar el vocabulario, con todas las palabras unicas.

In [ ]:
vocabulario = sorted(set(palabra for doc in tokens for palabra in doc))vocabulario

In [ ]:
len(vocabulario)

Paso 3: calcular TF (frecuencia de cada palabra dentro de su documento).

In [ ]:
def calcular_tf(doc, vocab):    return [doc.count(palabra) / len(doc) for palabra in vocab]

In [ ]:
tf_matriz = [calcular_tf(doc, vocabulario) for doc in tokens]tf_df = pd.DataFrame(tf_matriz, columns=vocabulario, index=['reporte_1', 'reporte_2', 'reporte_3'])tf_df

Paso 4: calcular DF (en cuantos documentos aparece cada palabra) e IDF.

In [ ]:
n_docs = len(tokens)df_terminos = {p: sum(1 for doc in tokens if p in doc) for p in vocabulario}idf = {p: np.log(n_docs / df_terminos[p]) for p in vocabulario}

In [ ]:
idf

Paso 5: TF x IDF, el resultado final.

In [ ]:
tfidf_matriz = tf_df.copy()for palabra in vocabulario:    tfidf_matriz[palabra] = tf_df[palabra] * idf[palabra]

In [ ]:
tfidf_matriz

## ConclusionesSe genero un dataset financiero con numpy y pandas, se entreno un arbol de decision y una red neuronal en pytorch para predecir la tendencia del mercado, se hicieron graficos para visualizar los datos y se aplico tokenizacion con TF-IDF sobre reportes de ejemplo.Repositorio: (pegar aqui el link de GitHub)